# Baseline Comparison + VAE–Gaussian Score Fusion

Input: `data/processed/windows_cc1/X_*.npy` (same whitened PCA features the VAE uses).

**Goal (base PR-AUC):** raise in-distribution PR-AUC above the VAE-alone ~0.60
**without** increasing `WINDOW_SIZE` (mid/long windows hurt `drift_cc2`).

Baselines (leak-free `cc1_val` thresholds):
1. **Gaussian** `||x||^2` in whitened PCA space (ID PR-AUC historically ~0.655)
2. **Isolation Forest**
3. **VAE** reconstruction MSE
4. **NEW — Score fusion** `α·z(VAE) + (1-α)·z(Gaussian)` and `max(z_VAE, z_G)`
   - Score z-normalization from **cc1_train** only
   - Deployed α fixed a priori at **0.5** (no test-label tuning)
   - Full α-grid reported for sensitivity; "best on cc1_test" flagged as diagnostic only


In [ ]:
import numpy as np
import pickle, os
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix,
)

BASE     = r'c:\Users\DELL\Documents\Claude\Projects\FYP\module3'
WIN_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')

SETS = ['cc1_train', 'cc1_val', 'cc1_test', 'drift_cc2']  # CC2-only (matches vae_eval)
DRIFT_SETS = ['drift_cc2']
EVAL_SETS = ['cc1_test', 'drift_cc2']

X, y = {}, {}
for name in SETS:
    X[name] = np.load(os.path.join(WIN_DIR, f'X_{name}.npy')).astype(np.float32)
    y[name] = np.load(os.path.join(WIN_DIR, f'y_{name}.npy'))
    print(f'  {name:10s}: {X[name].shape}  anomalies={int(y[name].sum()):,}')

vae_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
meta = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
CLIP = float(meta.get('clip', 20.0))
for name in SETS:
    X[name] = np.clip(X[name], -CLIP, CLIP).astype(np.float32)

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden2), nn.ReLU(),
            nn.Linear(hidden2, hidden1), nn.ReLU(),
            nn.Linear(hidden1, input_dim),
        )
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z):
        return self.decoder(z)
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval()
        mu, _ = self.encode(x)
        return ((self.decode(mu) - x) ** 2).mean(dim=1)

model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
model.eval()

scores_vae = {}
for name in SETS:
    with torch.no_grad():
        scores_vae[name] = model.anomaly_score(torch.from_numpy(X[name])).numpy()
print('VAE MSE scores computed for all sets.')
print('VAE reference eval loaded for comparison.')


## Baseline A — Gaussian / whitened-distance score

`score(x) = sum(x_i^2)`. No fitting needed — the PCA whitening already IS the
model. This tests whether the VAE's nonlinear reconstruction is doing anything
that simple per-component squared distance from the training mean couldn't.

In [2]:
def gaussian_score(Xarr):
    return (Xarr ** 2).sum(axis=1)

scores_gaussian = {name: gaussian_score(X[name]) for name in SETS}

print('cc1_train whitening sanity check (should be close to 0 mean / 1 std per component):')
print(f'  mean of per-component mean: {X["cc1_train"].mean():.4f}')
print(f'  mean of per-component std:  {X["cc1_train"].std():.4f}')


cc1_train whitening sanity check (should be close to 0 mean / 1 std per component):
  mean of per-component mean: 0.0000
  mean of per-component std:  1.0000


## Baseline B — Isolation Forest

Fit on `cc1_train` only (same training set the VAE uses), same whitened PCA
features. `n_estimators=100` (sklearn default-adjacent), `contamination='auto'`
(doesn't use labels — purely structural).

In [3]:
iso_forest = IsolationForest(n_estimators=100, contamination='auto', random_state=42, n_jobs=-1)
iso_forest.fit(X['cc1_train'])

# decision_function: higher = more normal. Negate so higher = more anomalous, matching every other score in this project.
scores_iso = {name: -iso_forest.decision_function(X[name]) for name in SETS}
print('Isolation Forest fit on cc1_train.')


Isolation Forest fit on cc1_train.


## Leak-free thresholds for each baseline (from `cc1_val` only, same discipline as `vae_eval.ipynb`)

In [4]:
thresh_gaussian = float(np.percentile(scores_gaussian['cc1_val'], 99))
thresh_iso      = float(np.percentile(scores_iso['cc1_val'], 99))
print(f'Gaussian baseline val_p99 threshold: {thresh_gaussian:.4f}')
print(f'Isolation Forest val_p99 threshold:  {thresh_iso:.4f}')


Gaussian baseline val_p99 threshold: 85.5405
Isolation Forest val_p99 threshold:  0.0375


## Full comparison: VAE vs. Gaussian vs. Isolation Forest, on every set

In [ ]:
def evaluate(scores, y_true, threshold):
    auc_roc = roc_auc_score(y_true, scores)
    auc_pr  = average_precision_score(y_true, scores)
    pred = scores > threshold
    return {
        'auc_roc': float(auc_roc), 'auc_pr': float(auc_pr),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'fpr': float(((pred) & (y_true == 0)).sum() / max(1, (y_true == 0).sum())),
    }

# CC2-only eval scope (matches current vae_cc1_eval.pkl)
EVAL_SETS = ['cc1_test', 'drift_cc2']
results = {'gaussian': {}, 'isolation_forest': {}, 'vae': {}}

print(f'{"set":12s} {"method":18s} {"AUC-ROC":>8s} {"AUC-PR":>8s} {"F1":>7s} {"Prec":>7s} {"Rec":>7s}')
for name in EVAL_SETS:
    results['gaussian'][name] = evaluate(scores_gaussian[name], y[name], thresh_gaussian)
    results['isolation_forest'][name] = evaluate(scores_iso[name], y[name], thresh_iso)
    # VAE metrics from saved eval when available; else recompute not done here
    if name in vae_eval.get('auc', {}):
        pr = vae_eval['precision_recall'][name]['val_p99']
        results['vae'][name] = {
            'auc_roc': vae_eval['auc'][name]['auc_roc'],
            'auc_pr': vae_eval['auc'][name]['auc_pr'],
            'precision': pr['precision'], 'recall': pr['recall'], 'f1': pr['f1'],
            'fpr': pr.get('fpr'),
        }
    for method in ['gaussian', 'isolation_forest', 'vae']:
        r = results[method][name]
        print(f'{name:12s} {method:18s} {r["auc_roc"]:8.4f} {r["auc_pr"]:8.4f} '
              f'{r["f1"]:7.3f} {r["precision"]:7.3f} {r["recall"]:7.3f}')


## Score fusion — VAE + Gaussian (raise base PR-AUC without window=60)

Gaussian already beats VAE on in-distribution PR-AUC (~0.655 vs ~0.601) while
VAE wins under drift. Fuse z-scored anomaly scores:

```
z_v = (mse_vae - μ_vae) / σ_vae     # μ,σ from cc1_train only
z_g = (||x||²  - μ_g)   / σ_g
fused_α = α·z_v + (1-α)·z_g
fused_max = max(z_v, z_g)
```

**Deployed rule (leak-free):** α = **0.5** fixed a priori + `val_p99` threshold
on fused scores from `cc1_val` only.

Also report α-grid and max-fusion. "Best α on cc1_test" is diagnostic only
(same honesty rule as oracle-ceiling — not a deployed threshold selector).


In [ ]:

# --- z-normalize using cc1_train score statistics only (leak-free) ---
mu_v, sig_v = float(scores_vae['cc1_train'].mean()), float(scores_vae['cc1_train'].std())
mu_g, sig_g = float(scores_gaussian['cc1_train'].mean()), float(scores_gaussian['cc1_train'].std())
sig_v = max(sig_v, 1e-8)
sig_g = max(sig_g, 1e-8)

def z_vae(s):
    return (s - mu_v) / sig_v

def z_gauss(s):
    return (s - mu_g) / sig_g

ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]  # 0=pure Gaussian, 1=pure VAE
DEPLOY_ALPHA = 0.5  # a priori — no test-label tuning

z_v = {name: z_vae(scores_vae[name]) for name in SETS}
z_g = {name: z_gauss(scores_gaussian[name]) for name in SETS}

fusion_scores = {}  # key -> {set -> scores}
for a in ALPHAS:
    fusion_scores[f'fuse_a{a}'] = {
        name: a * z_v[name] + (1.0 - a) * z_g[name] for name in SETS
    }
fusion_scores['fuse_max'] = {
    name: np.maximum(z_v[name], z_g[name]) for name in SETS
}

# Leak-free thresholds: val_p99 of each fused score on cc1_val
fusion_thresh = {
    key: float(np.percentile(sc['cc1_val'], 99)) for key, sc in fusion_scores.items()
}

print('=== Fusion α-grid (PR-AUC primary) ===')
print(f'{"set":12s} {"method":14s} {"PR-AUC":>8s} {"ROC-AUC":>8s} {"F1":>7s} {"Prec":>7s} {"Rec":>7s} {"FPR":>8s}')

fusion_results = {}
for key, sc in fusion_scores.items():
    fusion_results[key] = {}
    for name in EVAL_SETS:
        fusion_results[key][name] = evaluate(sc[name], y[name], fusion_thresh[key])

# Also keep single-model rows already in `results`
for name in EVAL_SETS:
    # ensure VAE row uses live scores if present (consistent with fusion)
    results['vae'][name] = evaluate(scores_vae[name], y[name], float(np.percentile(scores_vae['cc1_val'], 99)))
    for method in ['gaussian', 'isolation_forest', 'vae']:
        r = results[method][name]
        print(f'{name:12s} {method:14s} {r["auc_pr"]:8.4f} {r["auc_roc"]:8.4f} '
              f'{r["f1"]:7.3f} {r["precision"]:7.3f} {r["recall"]:7.3f} {r.get("fpr", float("nan")):8.4f}')
    for key in [f'fuse_a{a}' for a in ALPHAS] + ['fuse_max']:
        r = fusion_results[key][name]
        print(f'{name:12s} {key:14s} {r["auc_pr"]:8.4f} {r["auc_roc"]:8.4f} '
              f'{r["f1"]:7.3f} {r["precision"]:7.3f} {r["recall"]:7.3f} {r["fpr"]:8.4f}')
    print()

# Deployed fusion
deploy_key = f'fuse_a{DEPLOY_ALPHA}'
results['fusion_deployed'] = fusion_results[deploy_key]
results['fusion_max'] = fusion_results['fuse_max']
results['fusion_grid'] = fusion_results

# Diagnostic: best α on cc1_test PR-AUC (NOT leak-free for deployment claims)
best_a = max(ALPHAS, key=lambda a: fusion_results[f'fuse_a{a}']['cc1_test']['auc_pr'])
best_key = f'fuse_a{best_a}'
diag = fusion_results[best_key]['cc1_test']
dep = results['fusion_deployed']['cc1_test']
vae_id = results['vae']['cc1_test']['auc_pr']
gau_id = results['gaussian']['cc1_test']['auc_pr']

print('=== Deployed fusion (α=0.5, leak-free) ===')
print(f'  cc1_test  PR-AUC={dep["auc_pr"]:.4f}  F1={dep["f1"]:.3f}  '
      f'(VAE={vae_id:.4f}, Gaussian={gau_id:.4f})')
print(f'  drift_cc2 PR-AUC={results["fusion_deployed"]["drift_cc2"]["auc_pr"]:.4f}  '
      f'F1={results["fusion_deployed"]["drift_cc2"]["f1"]:.3f}')

print('\n=== Diagnostic only: best α on cc1_test PR-AUC (do NOT claim as deployed) ===')
print(f'  best α={best_a}  cc1_test PR-AUC={diag["auc_pr"]:.4f}  F1={diag["f1"]:.3f}')

# Success gates vs VAE alone
id_gain = dep['auc_pr'] - vae_id
drift_vae = results['vae']['drift_cc2']['auc_pr']
drift_fuse = results['fusion_deployed']['drift_cc2']['auc_pr']
print('\n=== Success check (deployed α=0.5) ===')
print(f'  ID PR-AUC gain vs VAE: {id_gain:+.4f}  (target: > 0, stretch: ID >= 0.65)')
print(f'  ID PR-AUC >= Gaussian? {dep["auc_pr"] >= gau_id - 1e-6}  '
      f'(Gaussian={gau_id:.4f}, fusion={dep["auc_pr"]:.4f})')
print(f'  Drift PR-AUC change vs VAE: {drift_fuse - drift_vae:+.4f}  '
      f'(prefer not << VAE)')
print(f'  PASS base goal (fusion ID PR-AUC > VAE ID): {dep["auc_pr"] > vae_id}')


## Save results

In [ ]:
out_path = os.path.join(MODEL_DIR, 'baseline_comparison.pkl')
save_payload = {
    'results': results,  # includes fusion_deployed, fusion_max, fusion_grid
    'deploy_alpha': DEPLOY_ALPHA,
    'alphas': ALPHAS,
    'fusion_thresholds': fusion_thresh,
    'score_norm': {
        'vae_mu': mu_v, 'vae_sigma': sig_v,
        'gauss_mu': mu_g, 'gauss_sigma': sig_g,
    },
    'diagnostic_best_alpha_on_cc1_test': best_a,
}
with open(out_path, 'wb') as f:
    pickle.dump(save_payload, f)
print(f'Saved -> {out_path}')
print(f'Deployed fusion α={DEPLOY_ALPHA}  ID PR-AUC={results["fusion_deployed"]["cc1_test"]["auc_pr"]:.4f}')


## How to read this

- **Gaussian vs VAE:** Gaussian often wins **in-distribution PR-AUC**; VAE wins under **drift**.
- **Fusion (α=0.5):** intended to raise **base PR-AUC** above VAE-alone (~0.60) toward
  Gaussian (~0.65) while retaining more drift ranking than pure Gaussian.
- If deployed fusion ID PR-AUC still &lt; 0.65, next step is architecture change
  (sequence AE at W=30) — **not** mid window sizes (45/50 collapse drift).
- "Best α on cc1_test" is diagnostic only; the thesis claim should use **α=0.5**.
